# Oracle DB Query Helper
엑셀 파일의 데이터를 읽어 Oracle DB IN절 조회 후 결과를 엑셀로 저장

In [ ]:
# 필요한 패키지 설치 (최초 1회)
# !pip install oracledb pandas openpyxl

In [ ]:
import oracledb
import pandas as pd
from datetime import datetime

## 1. Oracle 접속 정보 입력

In [ ]:
# ===== Oracle 접속 정보 =====
DB_HOST = "localhost"        # DB 호스트
DB_PORT = 1521               # DB 포트
DB_SERVICE = "ORCL"          # 서비스명 (SID)
DB_USER = "your_user"        # 사용자명
DB_PASSWORD = "your_password" # 비밀번호

In [ ]:
# 접속 테스트
dsn = f"{DB_HOST}:{DB_PORT}/{DB_SERVICE}"
conn = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=dsn)
print(f"Oracle 접속 성공: {dsn}")
conn.close()

## 2. 엑셀 파일 읽기

In [ ]:
# ===== 엑셀 파일 설정 =====
EXCEL_PATH = "input.xlsx"    # 읽을 엑셀 파일 경로
SHEET_NAME = 0               # 시트명 또는 인덱스 (0 = 첫번째 시트)
COLUMN_NAME = "ID"           # IN절에 넣을 컬럼명

df_input = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)
values = df_input[COLUMN_NAME].dropna().unique().tolist()
print(f"엑셀 로드 완료: 총 {len(values)}건 (중복 제거 후)")
df_input.head()

## 3. 쿼리 설정

In [ ]:
# ===== 쿼리 설정 =====
# {IN_CLAUSE} 자리에 IN절 값이 자동으로 들어갑니다.
QUERY_TEMPLATE = """
SELECT *
FROM YOUR_TABLE
WHERE YOUR_COLUMN IN ({IN_CLAUSE})
"""

CHUNK_SIZE = 1000  # IN절 최대 개수 (Oracle 제한: 1000개)

print(f"총 {len(values)}건 / {CHUNK_SIZE}건씩 = {(len(values) - 1) // CHUNK_SIZE + 1}회 조회 예정")

## 4. 쿼리 실행 (1000건씩 분할 조회 + 누적)

In [ ]:
results = []
total_chunks = (len(values) - 1) // CHUNK_SIZE + 1

conn = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=dsn)
cursor = conn.cursor()

try:
    for i in range(0, len(values), CHUNK_SIZE):
        chunk = values[i:i + CHUNK_SIZE]
        chunk_num = i // CHUNK_SIZE + 1

        # 바인드 변수 생성
        bind_names = [f":v{j}" for j in range(len(chunk))]
        bind_dict = {f"v{j}": val for j, val in enumerate(chunk)}

        query = QUERY_TEMPLATE.replace("{IN_CLAUSE}", ", ".join(bind_names))
        cursor.execute(query, bind_dict)

        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()
        
        if rows:
            df_chunk = pd.DataFrame(rows, columns=columns)
            results.append(df_chunk)

        print(f"  [{chunk_num}/{total_chunks}] 조회 완료 - IN절 {len(chunk)}건 → 결과 {len(rows)}건")

finally:
    cursor.close()
    conn.close()
    print("\nDB 접속 종료")

# 결과 누적 합치기
if results:
    df_result = pd.concat(results, ignore_index=True)
    print(f"\n전체 조회 결과: 총 {len(df_result)}건")
else:
    df_result = pd.DataFrame()
    print("\n조회 결과가 없습니다.")

In [ ]:
# 결과 미리보기
df_result.head(10)

## 5. 결과 엑셀 저장

In [ ]:
OUTPUT_PATH = f"output_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

df_result.to_excel(OUTPUT_PATH, index=False, engine="openpyxl")
print(f"저장 완료: {OUTPUT_PATH} ({len(df_result)}건)")